In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from pathlib import Path

from data.datasets import dataset_from_schema
from model.loader import load_model, load_thresholds
from model.model import MultitaskMLPWithQRegression
from model.predict import run_inference, predict_single
from predicates.interface import PredictionInterface, PredictionEntry, ContinuousRange
from predicates.compiler import compile_asp_constraint

In [3]:
checkpoint_path = "../models/latest/test.best.pt"

In [4]:
# Load the model and dataset
model, dataset, device = load_model(checkpoint_path)
print(f"Loaded model {type(model)}")

Loaded model <class 'model.model.MultitaskMLPWithQRegression'>


In [5]:
thresholds = load_thresholds(Path(checkpoint_path).resolve())
if thresholds is not None:
    print("Conformal thresholds loaded.")
else:
    print("No conformal thresholds found. Run calibrate.py on this checkpoint.")

No conformal thresholds found. Run calibrate.py on this checkpoint.


In [46]:
import numpy as np
from tqdm import tqdm

smallest_formula_space = np.inf
smallest_idx = None
for idx in tqdm(range(min(len(dataset), 1000))):
    x, _ = dataset[idx]
    pred = predict_single(x, model, dataset, thresholds=thresholds)
    formula_space = np.prod([hi - lo + 1 for lo, hi in pred.count_ranges.values()])
    if formula_space < smallest_formula_space:
        smallest_formula_space = formula_space
        smallest_idx = idx

print(f"Smallest formula space: {smallest_formula_space}")
print(f"Index of smallest formula space: {smallest_idx}")

100%|██████████| 1000/1000 [01:05<00:00, 15.21it/s]

Smallest formula space: 26910
Index of smallest formula space: 720


In [47]:
idx = 720
x, gt = dataset[idx]
pred = predict_single(x, model, dataset, thresholds=thresholds)
print(pred)

RecordPrediction
  counts:
    num_carbons: [0, 12]
    num_hydrogens: [0, 22]
    num_nitrogens: [0, 4]
    num_oxygens: [0, 5]
    num_phosphorus: 0
    num_sulfurs: [0, 2]
  continuous (83 features):
    alcohol: [-0.25, 1.12]
    aldehyde: 0
    aliphatic_ring_count: [-0.87, 3.03]
    amide: [-0.39, 2.14]
    amine: [-0.15, 1.85]
    aniline: [-0.17, 0.96]
    aromatic_atom_count: [-3.02, 11.19]
    aromatic_heteroatom_count: [-0.85, 3.65]
    aromatic_n_count: [-0.26, 3.23]
    aromatic_o_count: [-0.16, 0.78]
    aromatic_p_count: 0
    aromatic_ring: [-0.60, 1.64]
    aromatic_ring_count: [-0.53, 2.40]
    aromatic_s_count: 0
    assigned_chiral_center_count: [-0.21, 0.88]
    branch_point_count: [-1.09, 6.63]
    bridgehead_atom_count: [-0.40, 1.78]
    carboxylic_acid: [-0.07, 1.07]
    chiral_center_count: [-0.69, 2.90]
    connected_component_count: 1
    degree_1_atom_count: [0.01, 9.40]
    degree_2_atom_count: [-3.04, 10.93]
    degree_3_atom_count: [-1.28, 7.13]
    degre

In [77]:
# Print the unnormalized ground truth atom counts
from model.predict import _ELEMENT_TO_COUNT_KEY

record = dataset._load_record(idx)
feats = record["structural_features"]
atom_counts_gt = {}
for feat, key in _ELEMENT_TO_COUNT_KEY.items():
    atom_counts_gt[key] = int(feats.get(feat, 0))
print("Ground truth atom counts:")
for key, count in atom_counts_gt.items():
    print(f"{key}: {count}")
smiles = record["smiles"]
print(f"SMILES: {smiles}")

Ground truth atom counts:
num_carbons: 6
num_hydrogens: 12
num_nitrogens: 2
num_oxygens: 0
num_phosphorus: 0
num_sulfurs: 2
SMILES: CCNC(=S)C(=S)NCC


In [54]:
# Convert into a prediction interface object
interface = pred.to_interface()
for key, value in interface.root.items():
    print(f"{key}: {value}")

1: pred_set=[0] alpha=0.05 proba=[0.8703897297382355]
2: pred_set=[0] alpha=0.05 proba=[0.8759478628635406]
3: pred_set=[0] alpha=0.05 proba=[0.9074793606996536]
4: pred_set=[0] alpha=0.05 proba=[0.9058859273791313]
5: pred_set=[0] alpha=0.05 proba=[0.8845242708921432]
6: pred_set=[0] alpha=0.05 proba=[0.823864534497261]
7: pred_set=[0] alpha=0.05 proba=[0.9018167331814766]
8: pred_set=[0] alpha=0.05 proba=[0.8928489089012146]
9: pred_set=[0] alpha=0.05 proba=[0.8999388962984085]
10: pred_set=[0] alpha=0.05 proba=[0.8787518292665482]
11: pred_set=[0] alpha=0.05 proba=[0.8767824098467827]
12: pred_set=[0] alpha=0.05 proba=[0.804739773273468]
13: pred_set=[0] alpha=0.05 proba=[0.8765851780772209]
14: pred_set=[0] alpha=0.05 proba=[0.8909867033362389]
15: pred_set=[0] alpha=0.05 proba=[0.7948088496923447]
16: pred_set=[0] alpha=0.05 proba=[0.8413859307765961]
17: pred_set=[0] alpha=0.05 proba=[0.8954358920454979]
18: pred_set=[0] alpha=0.05 proba=[0.8966325148940086]
19: pred_set=[0] alph

In [55]:
asp_constraints, atom_counts = compile_asp_constraint(interface)
for c in asp_constraints:
    print(c)
print(atom_counts)


            % Non-aromatic N= only; aromatic ring N= excluded (Kekulé mismatch).
            has_n_eq_a :-
                double_bond_between(I, J),
                symbol(I, "N"),
                not aromatic_atom(I),
                I != J.
        

            has_q_ch2a :-
                any_bond_between(I, J),
                any_bond_between(J, K),
                heteroatom(I),
                symbol(J, "C"),
                free_bonds(J, 2),
                I != J, J != K, I != K.
        

        atom_degree(I, D) :-
            symbol(I, _),
            D = #count { J : heavy_neighbor(I, J) }.
    

        ring_size_8plus_count(N) :-
            N = #count { CYCLE : ring_size(CYCLE, S), S >= 8 }.
    

            has_qh_a_a_ch2_a :-
                heavy_neighbor(I, J),
                heavy_neighbor(J, K),
                heavy_neighbor(K, L),
                heavy_neighbor(L, M),
                qh_atom(I),
                ch2_atom(L),
                I != J, J != K,

In [ ]:
# Run ASP
from asp.bridge import solve

# Override atom counts
print(f"Solving for C{atom_counts_gt['num_carbons']}, H{atom_counts_gt['num_hydrogens']}, N{atom_counts_gt['num_nitrogens']}, O{atom_counts_gt['num_oxygens']}, S{atom_counts_gt['num_sulfurs']}")
%time candidates, _, _= solve(asp_constraints, atom_counts_gt, output_list_of_smiles=True, timeout=10)

print(f"Found {len(candidates)} candidate molecules.")
for smi in candidates:
    print(smi)

Solving for C0, H0, N0, O4, S2


In [57]:
# Molecular formula combos
n = 1
for lo, hi in pred.count_ranges.values():
    n *= (hi - lo + 1)
print(n, "combinations")

26910 combinations


In [76]:
import itertools

all_candidates = []
seen = set()

combos = itertools.product(*[range(lo, hi + 1) for lo, hi in pred.count_ranges.values()])
for combo in tqdm(combos):
    counts = dict(zip(pred.count_ranges, combo))
    interface = pred.to_interface(counts=counts)
    asp_constraints, atom_counts = compile_asp_constraint(interface)
    candidates, _, _ = solve(asp_constraints, atom_counts, output_list_of_smiles=True)
    for smi in candidates:
        if smi not in seen:
            seen.add(smi)
            all_candidates.append(smi)
print(f"{len(all_candidates)} unique candidates across all formulas")

14it [00:44,  3.20s/it]


KeyboardInterrupt: 